# Sales Analytics EDA
**aws-athena-quicksight-sales-analytics**

This notebook demonstrates the full local reporting-system:
1. Load cleaned Parquet datasets
2. Validate data quality
3. Exploratory Data Analysis (EDA)
4. Feature engineering
5. Export summary report

All cells are designed to run end-to-end after executing the Python scripts:
```bash
python datasets/generate_datasets.py
python python/clean_sales_data.py
```

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pyarrow.parquet as pq

# Plotting style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'

DATA_DIR = Path('../data/processed')

print('Libraries loaded successfully.')
print(f'Data directory: {DATA_DIR.resolve()}')

## 1. Load Cleaned Datasets

In [ ]:
def load_parquet(path: Path) -> pd.DataFrame:
    """Load a Parquet file or partitioned dataset directory."""
    if path.is_dir():
        return pq.read_table(str(path)).to_pandas()
    return pd.read_parquet(path)

customers = load_parquet(DATA_DIR / 'customers')
products  = load_parquet(DATA_DIR / 'products')
txn       = load_parquet(DATA_DIR / 'sales_transactions')

print(f'customers:         {len(customers):,} rows x {customers.shape[1]} cols')
print(f'products:          {len(products):,} rows x {products.shape[1]} cols')
print(f'sales_transactions:{len(txn):,} rows x {txn.shape[1]} cols')

In [ ]:
customers.head(3)

In [ ]:
products.head(3)

In [ ]:
txn.head(3)

## 2. Data Quality Validation

In [ ]:
def dq_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Generate a data quality summary for a DataFrame."""
    report = pd.DataFrame({
        'column':      df.columns,
        'dtype':       df.dtypes.values,
        'null_count':  df.isna().sum().values,
        'null_pct':    (df.isna().mean() * 100).round(2).values,
        'unique':      df.nunique().values,
    })
    print(f'\n=== {name} — Data Quality Report ===')
    return report

display(dq_report(customers, 'customers'))
display(dq_report(products, 'products'))
display(dq_report(txn, 'sales_transactions'))

In [ ]:
# Duplicate checks
print('Duplicate customer_id:   ', customers['customer_id'].duplicated().sum())
print('Duplicate product_id:    ', products['product_id'].duplicated().sum())
print('Duplicate transaction_id:', txn['transaction_id'].duplicated().sum())

# Referential integrity
orphan_cust = ~txn['customer_id'].isin(customers['customer_id'])
orphan_prod = ~txn['product_id'].isin(products['product_id'])
print('Orphan customer_id in txn:', orphan_cust.sum())
print('Orphan product_id in txn: ', orphan_prod.sum())

# Business rule: ship_date >= order_date
txn['order_date'] = pd.to_datetime(txn['order_date'].astype(str))
txn['ship_date']  = pd.to_datetime(txn['ship_date'].astype(str))
bad_ship = (txn['ship_date'] < txn['order_date']).sum()
print('ship_date < order_date:  ', bad_ship)

print('\nAll checks passed!' if all(v == 0 for v in [orphan_cust.sum(), orphan_prod.sum(), bad_ship]) else '\n⚠️  Issues found — review above.')

## 3. Exploratory Data Analysis

### 3.1 Revenue Overview

In [ ]:
total_revenue      = txn['net_revenue'].sum()
total_orders       = txn['transaction_id'].nunique()
unique_customers   = txn['customer_id'].nunique()
avg_order_value    = total_revenue / total_orders
avg_discount       = txn['discount'].mean()

print(f'Total Net Revenue:      ${total_revenue:>12,.2f}')
print(f'Total Orders:           {total_orders:>12,}')
print(f'Unique Customers:       {unique_customers:>12,}')
print(f'Avg Order Value:        ${avg_order_value:>12.2f}')
print(f'Avg Discount Rate:      {avg_discount*100:>11.1f}%')

In [ ]:
# Enrich transactions
txn_rich = txn.merge(
    products[['product_id', 'category', 'sub_category', 'unit_cost', 'brand']],
    on='product_id', how='left'
).merge(
    customers[['customer_id', 'segment', 'city']],
    on='customer_id', how='left'
)
txn_rich['gross_profit'] = txn_rich['net_revenue'] - txn_rich['unit_cost'] * txn_rich['quantity']
txn_rich['order_ym']     = txn_rich['order_date'].dt.to_period('M')
print('Enriched dataset shape:', txn_rich.shape)

### 3.2 Monthly Revenue Trend

In [ ]:
monthly = (
    txn_rich.groupby('order_ym')
    .agg(revenue=('net_revenue', 'sum'),
         gross_profit=('gross_profit', 'sum'),
         orders=('transaction_id', 'nunique'))
    .reset_index()
)
monthly['order_ym_str'] = monthly['order_ym'].astype(str)
monthly['margin_pct'] = monthly['gross_profit'] / monthly['revenue'] * 100

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(monthly['order_ym_str'], monthly['revenue'] / 1e3, color='steelblue', alpha=0.85, label='Revenue ($K)')
ax1.set_xlabel('Month')
ax1.set_ylabel('Revenue ($K)')
ax1.tick_params(axis='x', rotation=45)
ax1.set_title('Monthly Net Revenue and Gross Margin %')

ax2 = ax1.twinx()
ax2.plot(monthly['order_ym_str'], monthly['margin_pct'], color='darkorange', linewidth=2, marker='o', markersize=3, label='Margin %')
ax2.set_ylabel('Gross Margin %')
ax2.set_ylim(0, 60)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.tight_layout()
plt.show()

### 3.3 Revenue by Region

In [ ]:
region_rev = (
    txn_rich.groupby('region')
    .agg(revenue=('net_revenue', 'sum'),
         orders=('transaction_id', 'nunique'),
         customers=('customer_id', 'nunique'))
    .sort_values('revenue', ascending=False)
    .reset_index()
)
region_rev['aov'] = region_rev['revenue'] / region_rev['orders']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=region_rev, x='revenue', y='region', ax=axes[0], palette='Blues_r')
axes[0].set_xlabel('Total Revenue ($)')
axes[0].set_ylabel('')
axes[0].set_title('Revenue by Region')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

sns.barplot(data=region_rev, x='aov', y='region', ax=axes[1], palette='Oranges_r')
axes[1].set_xlabel('Average Order Value ($)')
axes[1].set_ylabel('')
axes[1].set_title('Avg Order Value by Region')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))

plt.tight_layout()
plt.show()

display(region_rev.style.format({'revenue': '${:,.0f}', 'aov': '${:.2f}'}))

### 3.4 Profitability by Category

In [ ]:
cat_profit = (
    txn_rich.groupby('category')
    .agg(revenue=('net_revenue', 'sum'),
         gross_profit=('gross_profit', 'sum'))
    .reset_index()
)
cat_profit['margin_pct'] = cat_profit['gross_profit'] / cat_profit['revenue'] * 100
cat_profit = cat_profit.sort_values('margin_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ca02c' if m > 25 else '#ff7f0e' if m > 15 else '#d62728' for m in cat_profit['margin_pct']]
bars = ax.barh(cat_profit['category'], cat_profit['margin_pct'], color=colors)
ax.set_xlabel('Gross Margin %')
ax.set_title('Gross Margin % by Product Category')
ax.axvline(20, linestyle='--', color='gray', linewidth=1, label='20% threshold')

for bar, val in zip(bars, cat_profit['margin_pct']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11)

ax.legend()
plt.tight_layout()
plt.show()

### 3.5 Customer Segment Analysis

In [ ]:
seg_rev = (
    txn_rich.groupby('segment')
    .agg(revenue=('net_revenue', 'sum'),
         customers=('customer_id', 'nunique'),
         orders=('transaction_id', 'nunique'))
    .reset_index()
)
seg_rev['ltv'] = seg_rev['revenue'] / seg_rev['customers']
seg_rev['rev_pct'] = seg_rev['revenue'] / seg_rev['revenue'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(seg_rev['rev_pct'], labels=seg_rev['segment'],
            autopct='%1.1f%%', startangle=140,
            colors=sns.color_palette('pastel'))
axes[0].set_title('Revenue Share by Customer Segment')

sns.barplot(data=seg_rev, x='segment', y='ltv', ax=axes[1], palette='Set2')
axes[1].set_ylabel('Average Customer LTV ($)')
axes[1].set_title('Avg LTV by Customer Segment')
for p in axes[1].patches:
    axes[1].annotate(f'${p.get_height():,.0f}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()
display(seg_rev.style.format({'revenue': '${:,.0f}', 'ltv': '${:.2f}', 'rev_pct': '{:.1f}%'}))

### 3.6 Discount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Discount distribution
disc_counts = txn_rich['discount'].value_counts().sort_index() * 100 / len(txn_rich)
axes[0].bar([f'{v*100:.0f}%' for v in disc_counts.index], disc_counts.values, color='steelblue')
axes[0].set_xlabel('Discount Rate')
axes[0].set_ylabel('% of Transactions')
axes[0].set_title('Discount Rate Distribution')

# Discount vs Gross Margin
disc_margin = (
    txn_rich.groupby('discount')
    .apply(lambda g: (g['gross_profit'].sum() / g['net_revenue'].sum()) * 100)
    .reset_index(name='margin_pct')
)
axes[1].plot([f'{v*100:.0f}%' for v in disc_margin['discount']],
             disc_margin['margin_pct'], marker='o', color='darkorange', linewidth=2)
axes[1].set_xlabel('Discount Rate')
axes[1].set_ylabel('Gross Margin %')
axes[1].set_title('Discount Rate vs Gross Margin %')
axes[1].axhline(0, linestyle='--', color='red', linewidth=1, label='Break-even')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# ── Customer LTV ──────────────────────────────────────────────────────────
customer_ltv = (
    txn_rich.groupby('customer_id')
    .agg(
        lifetime_value=('net_revenue', 'sum'),
        total_orders=('transaction_id', 'nunique'),
        first_order=('order_date', 'min'),
        last_order=('order_date', 'max'),
    )
    .reset_index()
)
customer_ltv['lifespan_days'] = (customer_ltv['last_order'] - customer_ltv['first_order']).dt.days
customer_ltv['avg_order_value'] = customer_ltv['lifetime_value'] / customer_ltv['total_orders']

# ── RFM Scores ────────────────────────────────────────────────────────────
reference_date = txn_rich['order_date'].max()
customer_ltv['recency_days'] = (reference_date - customer_ltv['last_order']).dt.days

customer_ltv['r_score'] = pd.qcut(customer_ltv['recency_days'],  q=5, labels=[5,4,3,2,1]).astype(int)
customer_ltv['f_score'] = pd.qcut(customer_ltv['total_orders'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
customer_ltv['m_score'] = pd.qcut(customer_ltv['lifetime_value'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
customer_ltv['rfm_total'] = customer_ltv['r_score'] + customer_ltv['f_score'] + customer_ltv['m_score']

def rfm_label(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    if r >= 3 and f >= 3:
        return 'Loyal Customers'
    if r >= 4 and f <= 2:
        return 'Recent Customers'
    if r <= 2 and f >= 4 and m >= 4:
        return 'At Risk'
    if r <= 2 and f <= 2:
        return 'Lost'
    return 'Potential Loyalists'

customer_ltv['rfm_segment'] = customer_ltv.apply(rfm_label, axis=1)

print('Customer LTV + RFM feature set shape:', customer_ltv.shape)
customer_ltv.head(5)

In [ ]:
# RFM segment distribution
rfm_dist = customer_ltv['rfm_segment'].value_counts().reset_index()
rfm_dist.columns = ['rfm_segment', 'count']

fig, ax = plt.subplots(figsize=(10, 5))
palette = {'Champions': '#2ca02c', 'Loyal Customers': '#1f77b4',
           'Recent Customers': '#17becf', 'Potential Loyalists': '#9467bd',
           'At Risk': '#ff7f0e', 'Lost': '#d62728'}
sns.barplot(data=rfm_dist, x='rfm_segment', y='count', ax=ax,
            palette=[palette.get(s, 'gray') for s in rfm_dist['rfm_segment']])
ax.set_xlabel('RFM Segment')
ax.set_ylabel('Customer Count')
ax.set_title('Customer Distribution by RFM Segment')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width()/2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# LTV distribution
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(customer_ltv['lifetime_value'], bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(customer_ltv['lifetime_value'].median(), color='orange', linestyle='--', linewidth=2,
           label=f'Median: ${customer_ltv["lifetime_value"].median():,.0f}')
ax.axvline(customer_ltv['lifetime_value'].quantile(0.90), color='red', linestyle='--', linewidth=2,
           label=f'90th pct: ${customer_ltv["lifetime_value"].quantile(0.90):,.0f}')
ax.set_xlabel('Customer Lifetime Value ($)')
ax.set_ylabel('Customer Count')
ax.set_title('Customer LTV Distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Year-over-Year Comparison

In [ ]:
yearly = (
    txn_rich.groupby(['year', 'region'])
    .agg(revenue=('net_revenue', 'sum'))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for region, grp in yearly.groupby('region'):
    ax.plot(grp['year'], grp['revenue'] / 1e3, marker='o', linewidth=2, label=region)

ax.set_xlabel('Year')
ax.set_ylabel('Revenue ($K)')
ax.set_title('Annual Revenue by Region (2020–2023)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}K'))
plt.tight_layout()
plt.show()

## 6. Export Summary Report

In [ ]:
import json
from datetime import date

output_dir = Path('../outputs')
output_dir.mkdir(parents=True, exist_ok=True)

report = {
    'generated_at': date.today().isoformat(),
    'customers': {
        'total_customers':        int(len(customers)),
        'segment_distribution':   customers['segment'].value_counts().to_dict(),
        'region_distribution':    customers['region'].value_counts().to_dict(),
    },
    'products': {
        'total_products':          int(len(products)),
        'category_counts':         products['category'].value_counts().to_dict(),
        'avg_margin_by_category':  products.groupby('category')['margin_pct'].mean().round(2).to_dict(),
    },
    'transactions': {
        'total_transactions':      int(len(txn)),
        'total_revenue':           round(float(txn['net_revenue'].sum()), 2),
        'avg_order_value':         round(float(txn['net_revenue'].mean()), 2),
        'avg_discount_rate':       round(float(txn['discount'].mean()), 4),
        'revenue_by_year':         txn_rich.groupby('year')['net_revenue'].sum().round(2).to_dict(),
        'revenue_by_region':       txn_rich.groupby('region')['net_revenue'].sum().round(2).to_dict(),
    },
    'rfm_segments': customer_ltv['rfm_segment'].value_counts().to_dict(),
}

report_path = output_dir / 'summary_report.json'
with report_path.open('w') as f:
    json.dump(report, f, indent=2, default=str)

print(f'Summary report written → {report_path}')
print(json.dumps(report['transactions'], indent=2, default=str))

## 7. Pipeline Completion

All cells completed successfully. The processed datasets are ready for upload to S3 and query via Amazon Athena.

**Next steps:**
```bash
# Upload to S3
aws s3 sync data/processed/ s3://YOUR-BUCKET/sales-analytics/processed/

# Run Athena SQL files in order (01–11)
# Connect QuickSight → SPICE dataset → Build executive reportings
```